In [11]:
import pandas as pd
import mlflow

In [12]:
from pathlib import Path

DATA_PATH = Path("./Data")
RAW_DATA_PATH = DATA_PATH / "raw" / "Fraud_Data.csv"
PROCESSED_DATA_PATH = DATA_PATH / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [13]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Fraud Detection")
mlflow.start_run(run_name="FeatureEncoding")

<ActiveRun: >

# Load Data with correct Types

In [14]:
df = pd.read_csv(
    PROCESSED_DATA_PATH / "1_feature_engineering_result.csv",
    parse_dates=["signup_time", "purchase_time"]
)

In [15]:
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,time_velocity,ip_user_share_count,device_user_share_count,day,hour,age_manual_binned,purchase_value_manual_binned
0,286057,2015-01-01 00:00:42,2015-03-25 11:33:06,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,0,7212744.0,0,0,25,11,2,0
1,309557,2015-01-01 00:00:43,2015-01-01 00:00:44,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,1,1.0,1,1,1,0,2,0
2,124539,2015-01-01 00:00:44,2015-01-01 00:00:45,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,1,1.0,2,2,1,0,2,0
3,161246,2015-01-01 00:00:45,2015-01-01 00:00:46,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,1,1.0,3,3,1,0,2,0
4,356414,2015-01-01 00:00:46,2015-01-01 00:00:47,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,1,1.0,4,4,1,0,2,0


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   user_id                       151112 non-null  int64         
 1   signup_time                   151112 non-null  datetime64[ns]
 2   purchase_time                 151112 non-null  datetime64[ns]
 3   purchase_value                151112 non-null  int64         
 4   device_id                     151112 non-null  object        
 5   source                        151112 non-null  object        
 6   browser                       151112 non-null  object        
 7   sex                           151112 non-null  object        
 8   age                           151112 non-null  int64         
 9   ip_address                    151112 non-null  object        
 10  class                         151112 non-null  int64         
 11  time_velocity

# OHE

In [17]:
dummies = pd.get_dummies(
    df[["source", "browser", "sex"]], drop_first=False
)
df = df.join(dummies)


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 28 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   user_id                       151112 non-null  int64         
 1   signup_time                   151112 non-null  datetime64[ns]
 2   purchase_time                 151112 non-null  datetime64[ns]
 3   purchase_value                151112 non-null  int64         
 4   device_id                     151112 non-null  object        
 5   source                        151112 non-null  object        
 6   browser                       151112 non-null  object        
 7   sex                           151112 non-null  object        
 8   age                           151112 non-null  int64         
 9   ip_address                    151112 non-null  object        
 10  class                         151112 non-null  int64         
 11  time_velocity

# Target Encoding and Frequency Encoding
* This need to know the Train Test split. Because We already removed all the unwanted and issue related rows. We can predetermine the split point(We already ran one iteration of the DS life cycle. So we take 80 20 split based on time based)

In [19]:
# Sort chronologically
df = df.sort_values("signup_time")

# 80% oldest data -> train
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

In [20]:
enc_maps = {}

for col in ["source", "browser"]:
    freq = train_df[col].value_counts(normalize=True)
    enc_maps[col] = freq

    train_df[f"{col}_fr_enc"] = train_df[col].map(freq)
    test_df[f"{col}_fr_enc"] = test_df[col].map(freq).fillna(0)
    
mlflow.log_dict(enc_maps, "frequency_encoding_maps.json")

C:\Users\trixr\AppData\Local\Temp\ipykernel_25484\3144321173.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[f"{col}_fr_enc"] = train_df[col].map(freq)
C:\Users\trixr\AppData\Local\Temp\ipykernel_25484\3144321173.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f"{col}_fr_enc"] = test_df[col].map(freq).fillna(0)
C:\Users\trixr\AppData\Local\Temp\ipykernel_25484\3144321173.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

In [21]:
enc_maps = {}

for col in ["source", "browser"]:
    
    target_mean = train_df.groupby(col)["class"].mean()
    enc_maps[col] = target_mean.to_dict()

    train_df[f"{col}_targ_enc"] = train_df[col].map(target_mean)
    test_df[f"{col}_targ_enc"] = test_df[col].map(target_mean)

mlflow.log_dict(enc_maps, "target_encoding_maps.json")

C:\Users\trixr\AppData\Local\Temp\ipykernel_25484\2004535801.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[f"{col}_targ_enc"] = train_df[col].map(target_mean)
C:\Users\trixr\AppData\Local\Temp\ipykernel_25484\2004535801.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f"{col}_targ_enc"] = test_df[col].map(target_mean)
C:\Users\trixr\AppData\Local\Temp\ipykernel_25484\2004535801.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usi

# Save CSV

In [22]:
df = pd.concat([train_df, test_df], axis=0, ignore_index=True)


In [23]:
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,...,browser_FireFox,browser_IE,browser_Opera,browser_Safari,sex_F,sex_M,source_fr_enc,browser_fr_enc,source_targ_enc,browser_targ_enc
0,286057,2015-01-01 00:00:42,2015-03-25 11:33:06,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
1,309557,2015-01-01 00:00:43,2015-01-01 00:00:44,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
2,124539,2015-01-01 00:00:44,2015-01-01 00:00:45,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
3,161246,2015-01-01 00:00:45,2015-01-01 00:00:46,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
4,356414,2015-01-01 00:00:46,2015-01-01 00:00:47,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798


In [24]:

df.to_csv(
    PROCESSED_DATA_PATH / "2_feature_encoding_result.csv",
    index=False)

In [25]:
from collections import defaultdict

# Build a string report
report_lines = []
report_lines.append("DataFrame Summary")
report_lines.append("=" * 50)
report_lines.append(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
report_lines.append("\nSummary of Feature Encoding Stage:")
report_lines.append(
"""
 Only did the OHE for all the categorical columns. All were nominal.
 But there can be few steps that can be improve here,
 1. adding Target based encoding or frequency based : We can't do this now because it can cause data leakage.
 2. When training linear models one of the each categrical feature col can be drop for prevent multicollinearity issue.
 3. We did the Target and frequency encoding too but under assumption that where the split happen.
"""
)
report_str = "\n".join(report_lines)

# Log to MLflow (within an active run)
mlflow.log_text(report_str, "f_encoding/dataframe_summary.txt")

In [26]:
mlflow.end_run()

🏃 View run FeatureEncoding at: http://localhost:5000/#/experiments/1/runs/fb0e1bd39c8d494a91eb8e923a877d41
🧪 View experiment at: http://localhost:5000/#/experiments/1
